In [1]:
import ee, json
from kaggle_secrets import UserSecretsClient
key = UserSecretsClient().get_secret("EE_KEY_V2")
email = json.loads(key)["client_email"]
ee.Initialize(ee.ServiceAccountCredentials(email, key_data=key), project="lulc-sar-v2-513720")
print("OK:", email, ee.Number(1).add(1).getInfo())
print(__import__("os").listdir("/kaggle/input"))

OK: colab-ee@lulc-sar-v2-513720.iam.gserviceaccount.com 2
['datasets']


In [2]:
import glob, pandas as pd

f = glob.glob('/kaggle/input/**/places_final_FROZEN.csv', recursive=True)
print(f)
df = pd.read_csv(f[0])
print(df.shape); print(df.columns.tolist()); print(df.head(3).to_string())
for c in df.columns:
    if c.lower() in ('split', 'group', 'run'):
        print(df[c].value_counts().to_string())

lat = float(df[[c for c in df.columns if c.lower() in ('lat', 'latitude')][0]].iloc[0])
lon = float(df[[c for c in df.columns if c.lower() in ('lon', 'lng', 'longitude')][0]].iloc[0])
print('check place:', lat, lon)
rect = ee.Geometry.Point([lon, lat]).buffer(5000).bounds()

def hist(img):
    return img.reduceRegion(ee.Reducer.frequencyHistogram(), rect, 10, maxPixels=1e8).getInfo()

for v in ['v100', 'v200']:
    try:
        w = ee.ImageCollection(f'ESA/WorldCover/{v}').first()
        print('WorldCover', v, w.bandNames().getInfo(), hist(w.select('Map')))
    except Exception as e:
        print('WorldCover', v, 'ERROR', str(e)[:200])

try:
    esri = ee.ImageCollection('projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS')
    print('ESRI size', esri.size().getInfo())
    print('ESRI bands', esri.first().bandNames().getInfo())
    print('ESRI props', esri.first().toDictionary().getInfo())
    for y in [2020, 2021, 2025]:
        im = esri.filterDate(f'{y}-01-01', f'{y+1}-01-01').filterBounds(rect).mosaic().select([0])
        print('ESRI', y, hist(im))
except Exception as e:
    print('ESRI ERROR', str(e)[:300])

try:
    for y in [2020, 2021]:
        dw = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filterBounds(rect)
              .filterDate(f'{y}-01-01', f'{y+1}-01-01').select('label').mode())
        print('DW', y, hist(dw))
except Exception as e:
    print('DW ERROR', str(e)[:200])

['/kaggle/input/datasets/vinayybg/lulc-meta/places_final_FROZEN.csv']
(1136, 8)
['id', 'name', 'lat', 'lon', 'continent', 'group', 'run', 'split']
   id        name    lat   lon continent group  run  split
0   1  Heidelberg  49.40  8.69    Europe  land    1  train
1   2       Milan  45.46  9.19    Europe  land    1  train
2   3    LochNess  57.30 -4.45    Europe  land    1  train
group
land        1000
city          41
coastal       30
wetland       21
polar         15
ocean         12
lake          10
mangrove       7
run
9    136
1    125
2    125
4    125
3    125
5    125
6    125
7    125
8    125
split
train       894
test        148
val          82
excluded     12
check place: 49.4 8.69
WorldCover v100 ['Map'] {'Map': {'10': 802390.5843137358, '20': 1, '30': 156317.54117647017, '40': 241727.59215686368, '50': 292724.078431378, '60': 13601.960784313713, '80': 24707.25882352941, '90': 9}}
WorldCover v200 ['Map'] {'Map': {'10': 798721.7647058936, '30': 149585.59999999945, '40': 221

In [3]:
import os, io, math, time, requests
import numpy as np, pandas as pd, rasterio

OUT = "/kaggle/working/v3_pilot"
START_Y, START_M, N_MONTHS = 2020, 1, 24
KM, P = 10, 256
PILOT_IDS = [1]
LABELS = ["wc2020", "wc2021", "esri2020", "esri2021", "dw2020", "dw2021"]
os.makedirs(OUT, exist_ok=True)

df = pd.read_csv("/kaggle/input/datasets/vinayybg/lulc-meta/places_final_FROZEN.csv")
df = df[df.split != "excluded"]
print("usable places:", len(df))
rows = list(df[df.id.isin(PILOT_IDS)].itertuples())

def utm(lat, lon):
    return f"EPSG:{(32600 if lat >= 0 else 32700) + int((lon + 180)//6) + 1}"

def box(lat, lon):
    dlat = KM/2/111.32
    dlon = KM/2/(111.32*math.cos(math.radians(lat)))
    return ee.Geometry.Rectangle([lon-dlon, lat-dlat, lon+dlon, lat+dlat])

def fetch(img, region, crs, tries=4):
    err = None
    for k in range(tries):
        try:
            url = img.getDownloadURL({"region": region, "scale": 10, "crs": crs, "format": "GEO_TIFF"})
            r = requests.get(url, timeout=300); r.raise_for_status()
            with rasterio.open(io.BytesIO(r.content)) as src:
                return src.read()
        except Exception as e:
            err = e; time.sleep(4*(k+1))
    print("   fetch failed:", str(err)[:200])
    return None

def label_image(region):
    wc20 = ee.ImageCollection("ESA/WorldCover/v100").filterBounds(region).mosaic().select("Map")
    wc21 = ee.ImageCollection("ESA/WorldCover/v200").filterBounds(region).mosaic().select("Map")
    esri = ee.ImageCollection("projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS").filterBounds(region)
    e20 = esri.filterDate("2020-01-01", "2021-01-01").mosaic().select([0])
    e21 = esri.filterDate("2021-01-01", "2022-01-01").mosaic().select([0])
    dw = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(region)
    d20 = dw.filterDate("2020-01-01", "2021-01-01").select("label").mode()
    d21 = dw.filterDate("2021-01-01", "2022-01-01").select("label").mode()
    return ee.Image.cat([i.toByte() for i in [wc20, wc21, e20, e21, d20, d21]]).rename(LABELS)

def patches(arr, nr, nc):
    return [arr[:, i*P:(i+1)*P, j*P:(j+1)*P] for i in range(nr) for j in range(nc)]

def do_place(row):
    pid = int(row.id); folder = f"{OUT}/{pid:04d}"
    os.makedirs(folder, exist_ok=True)
    region, crs = box(row.lat, row.lon), utm(row.lat, row.lon)
    t0 = time.time()
    lab = fetch(label_image(region), region, crs)
    if lab is None:
        return pid, "no label", None
    nr, nc = lab.shape[1]//P, lab.shape[2]//P
    lp = np.stack(patches(lab, nr, nc)).astype("uint8")
    np.savez(f"{folder}/labels.npz", y=lp, names=np.array(LABELS), nr=nr, nc=nc)
    t_lab = time.time() - t0
    n_ok = n_empty = n_fail = 0
    for i in range(N_MONTHS):
        m = START_M - 1 + i; y, mo = START_Y + m//12, m % 12 + 1
        s = ee.Date.fromYMD(y, mo, 1)
        col = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(region)
               .filterDate(s, s.advance(1, "month"))
               .filter(ee.Filter.eq("instrumentMode", "IW"))
               .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
               .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))
        if col.size().getInfo() == 0:
            n_empty += 1; continue
        arr = fetch(col.select(["VV", "VH"]).median().float(), region, crs)
        if arr is None or arr.shape[1] < nr*P or arr.shape[2] < nc*P:
            n_fail += 1; continue
        np.savez(f"{folder}/{y}-{mo:02d}.npz", x=np.stack(patches(arr, nr, nc)).astype("float16"))
        n_ok += 1
    secs = time.time() - t0
    mb = sum(os.path.getsize(f"{folder}/{f}") for f in os.listdir(folder)) / 1e6
    open(f"{folder}/DONE", "w").write(f"{n_ok} months")
    return pid, dict(patches=len(lp), months_ok=n_ok, months_empty=n_empty, months_failed=n_fail,
                     label_sec=round(t_lab, 1), total_sec=round(secs, 1), size_MB=round(mb, 1)), lp

for row in rows:
    pid, info, lp = do_place(row)
    print(pid, row.name, info)
    if lp is not None:
        for k, nme in enumerate(LABELS):
            u, c = np.unique(lp[:, k], return_counts=True)
            print(f"   {nme}:", {int(a): round(100*b/c.sum(), 1) for a, b in zip(u, c)})

usable places: 1124
1 Heidelberg {'patches': 9, 'months_ok': 24, 'months_empty': 0, 'months_failed': 0, 'label_sec': 6.8, 'total_sec': 181.2, 'size_MB': 60.2}
   wc2020: {10: np.float64(44.5), 20: np.float64(0.0), 30: np.float64(10.0), 40: np.float64(15.8), 50: np.float64(26.2), 60: np.float64(1.3), 80: np.float64(2.2), 90: np.float64(0.0)}
   wc2021: {10: np.float64(44.2), 30: np.float64(9.5), 40: np.float64(14.2), 50: np.float64(30.0), 60: np.float64(0.1), 80: np.float64(2.1)}
   esri2020: {1: np.float64(2.9), 2: np.float64(27.5), 4: np.float64(0.1), 5: np.float64(19.7), 7: np.float64(48.9), 11: np.float64(0.8)}
   esri2021: {1: np.float64(2.9), 2: np.float64(27.3), 4: np.float64(0.1), 5: np.float64(20.4), 7: np.float64(48.7), 11: np.float64(0.6)}
   dw2020: {0: np.float64(2.8), 1: np.float64(31.5), 2: np.float64(0.3), 3: np.float64(0.0), 4: np.float64(19.3), 5: np.float64(0.0), 6: np.float64(45.9), 7: np.float64(0.0)}
   dw2021: {0: np.float64(3.0), 1: np.float64(29.8), 2: np.float6

In [4]:
import numpy as np, glob, os
folder = "/kaggle/working/v3_pilot/0001"
files = sorted(glob.glob(f"{folder}/20*.npz"))
print(len(files), "month files:", os.path.basename(files[0]), "to", os.path.basename(files[-1]))
L = np.load(f"{folder}/labels.npz")
print("labels", L["y"].shape, L["y"].dtype, list(L["names"]), int(L["nr"]), int(L["nc"]))
bad = 0
for f in files:
    x = np.load(f)["x"]; xf = x.astype("float32")
    nan = float(np.isnan(xf).mean()); inf = float(np.isinf(xf).mean()); zero = float((xf == 0).mean())
    bad += (nan > 0 or inf > 0 or zero > 0)
    print(os.path.basename(f), x.shape, x.dtype,
          f"VV {xf[:,0].mean():.1f}  VH {xf[:,1].mean():.1f}  nan {nan:.4f} inf {inf:.4f} zero {zero:.4f}")
print("files with nan/inf/zero:", bad)

24 month files: 2020-01.npz to 2021-12.npz
labels (9, 6, 256, 256) uint8 [np.str_('wc2020'), np.str_('wc2021'), np.str_('esri2020'), np.str_('esri2021'), np.str_('dw2020'), np.str_('dw2021')] 3 3
2020-01.npz (9, 2, 256, 256) float16 VV -8.4  VH -15.5  nan 0.0000 inf 0.0000 zero 0.0000
2020-02.npz (9, 2, 256, 256) float16 VV -7.8  VH -14.9  nan 0.0000 inf 0.0000 zero 0.0000
2020-03.npz (9, 2, 256, 256) float16 VV -8.3  VH -15.4  nan 0.0000 inf 0.0000 zero 0.0000
2020-04.npz (9, 2, 256, 256) float16 VV -9.2  VH -16.1  nan 0.0000 inf 0.0000 zero 0.0000
2020-05.npz (9, 2, 256, 256) float16 VV -9.0  VH -16.1  nan 0.0000 inf 0.0000 zero 0.0000
2020-06.npz (9, 2, 256, 256) float16 VV -8.5  VH -15.4  nan 0.0000 inf 0.0000 zero 0.0000
2020-07.npz (9, 2, 256, 256) float16 VV -8.6  VH -15.8  nan 0.0000 inf 0.0000 zero 0.0000
2020-08.npz (9, 2, 256, 256) float16 VV -8.5  VH -15.8  nan 0.0000 inf 0.0000 zero 0.0000
2020-09.npz (9, 2, 256, 256) float16 VV -8.7  VH -16.0  nan 0.0000 inf 0.0000 zero 0